In [1]:
import os
import sys

python_executable = sys.executable

os.environ["PYSPARK_PYTHON"] = python_executable
os.environ["PYSPARK_DRIVER_PYTHON"] = python_executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

print("Python:", python_executable)

Python: c:\Users\CES\Documents\DATA_ENGINEER\CURSO_SPARK\.venv\Scripts\python.exe


# 02 - Creación de DataFrames en PySpark

## 1. Objetivo del tema

El objetivo de este notebook es aprender las diferentes formas de crear un
DataFrame en PySpark y comprender cómo Spark determina la estructura de los
datos.

Al finalizar este tema estaré en capacidad de:

- Crear un DataFrame manualmente con `spark.createDataFrame()`.
- Leer un archivo CSV mediante `spark.read.csv()`.
- Comprender qué es el esquema o `schema` de un DataFrame.
- Permitir que Spark infiera automáticamente los tipos de datos mediante
  `inferSchema`.
- Indicar que un archivo CSV contiene nombres de columnas mediante `header`.
- Definir explícitamente un esquema utilizando `StructType` y `StructField`.
- Diferenciar entre crear un DataFrame desde objetos de Python y cargarlo desde
  una fuente externa.
- Reconocer cuándo conviene utilizar inferencia de esquema y cuándo es mejor
  definirlo manualmente.

---

## Relación con mi experiencia previa

### Paralelo con SQL y Teradata

Un DataFrame puede entenderse inicialmente como una estructura tabular formada
por filas y columnas con nombre.

Conceptualmente se parece a una tabla relacional:

| Teradata o SQL | PySpark |
|---|---|
| Tabla | DataFrame |
| Registro o fila | Row |
| Columna | Column |
| Definición de columnas y tipos | Schema |
| `CREATE TABLE` | Definición explícita mediante `StructType` |
| `SELECT` | Operaciones sobre el DataFrame |

Sin embargo, un DataFrame no es exactamente una tabla física.

Una tabla de Teradata normalmente se encuentra almacenada de forma persistente
en disco. Un DataFrame puede representar datos leídos desde un archivo, una
tabla, una consulta o una estructura de Python y puede existir solamente durante
la ejecución del proceso de Spark.

### Diferencia frente al Primary Index de Teradata

El esquema de un DataFrame define:

- nombres de columnas;
- tipos de datos;
- posibilidad de aceptar valores nulos;
- estructura lógica de los registros.

El esquema no define cómo se distribuyen físicamente los datos.

Esto es diferente al Primary Index de Teradata, que participa en la distribución
de las filas entre los AMP.

En Spark, la distribución se relaciona con las particiones del DataFrame. Este
tema se estudiará más adelante en el notebook de particiones y evaluación
perezosa.

### Paralelo con Informatica PowerCenter

Cuando se importa una fuente en PowerCenter, se crea una definición con:

- nombres de los campos;
- tipos de datos;
- longitudes;
- precisión;
- posibilidad de aceptar valores nulos.

En Spark, esta información corresponde conceptualmente al `schema`.

| Informatica PowerCenter | PySpark |
|---|---|
| Source Definition | Schema |
| Puerto de entrada | Columna |
| Tipo de dato del puerto | DataType |
| Fuente de archivo plano | `spark.read.csv()` |
| Filas que circulan por un mapping | Filas de un DataFrame |
| Metadata importada de la fuente | Esquema inferido o explícito |

La diferencia es que Spark puede ejecutar las transformaciones de forma
distribuida entre diferentes procesos y máquinas.

---

## Resultado esperado

Al terminar este notebook debo poder explicar:

> Un DataFrame es una estructura distribuida organizada en columnas con nombre.
> Puede crearse desde objetos de Python o cargarse desde una fuente externa.
> Todo DataFrame posee un esquema que define los nombres, tipos y nulabilidad de
> sus columnas. Spark puede inferir ese esquema, aunque en procesos productivos
> suele ser conveniente definirlo explícitamente cuando conocemos la estructura
> de la fuente.

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("02_creacion_dataframe")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.pyspark.python", python_executable)
    .config("spark.pyspark.driver.python", python_executable)
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

In [3]:
print("Python del notebook:", sys.executable)
print("Python del worker:", spark.sparkContext.pythonExec)
print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)

Python del notebook: c:\Users\CES\Documents\DATA_ENGINEER\CURSO_SPARK\.venv\Scripts\python.exe
Python del worker: c:\Users\CES\Documents\DATA_ENGINEER\CURSO_SPARK\.venv\Scripts\python.exe
Spark: 4.2.0
Master: local[2]


## 2. Creación manual de un DataFrame

### ¿Qué significa crear un DataFrame manualmente?

Crear un DataFrame manualmente significa proporcionar los datos directamente
desde Python, en lugar de leerlos desde un archivo, una tabla o una fuente
externa.

Para hacerlo utilizamos:

```python
spark.createDataFrame(data, schema)
```

El método `createDataFrame()` pertenece a la `SparkSession` y convierte una
estructura de datos compatible en un DataFrame de PySpark.

Los datos pueden provenir, entre otras fuentes, de:

- listas de tuplas;
- listas de diccionarios;
- objetos `Row`;
- RDD;
- DataFrames de pandas;
- tablas de PyArrow.

En este notebook comenzaremos con listas de tuplas porque permiten comprender
claramente la relación entre una fila y el esquema del DataFrame.

---

### Sintaxis básica

```python
df = spark.createDataFrame(data, schema)
```

Donde:

- `df` es la variable que almacenará el DataFrame.
- `spark` es la `SparkSession`.
- `data` contiene las filas.
- `schema` define la estructura de las columnas.

El parámetro `schema` puede proporcionarse de diferentes maneras:

1. Una lista con los nombres de las columnas.
2. Un esquema explícito con `StructType` y `StructField`.
3. Puede omitirse para permitir que Spark intente inferir la estructura.

---

### Ejemplo conceptual

```python
data = [
    (1, "Ana", 720),
    (2, "Carlos", 680),
    (3, "Laura", 750)
]
```

En este caso:

- cada tupla representa una fila;
- cada posición dentro de la tupla representa una columna;
- todas las filas deben mantener el mismo orden lógico.

La primera tupla:

```python
(1, "Ana", 720)
```

representa una fila con tres valores:

| Posición | Valor | Significado |
|---:|---|---|
| 1 | `1` | identificador del cliente |
| 2 | `"Ana"` | nombre del cliente |
| 3 | `720` | puntaje crediticio |

Para asignar nombres a esas posiciones podemos utilizar:

```python
columns = ["customer_id", "customer_name", "credit_score"]
```

Y crear el DataFrame:

```python
df = spark.createDataFrame(data, columns)
```

---

### Paralelo con SQL y Teradata

Esta estructura de Python:

```python
data = [
    (1, "Ana", 720),
    (2, "Carlos", 680)
]
```

puede relacionarse conceptualmente con:

```sql
INSERT INTO customers
VALUES
    (1, 'Ana', 720),
    (2, 'Carlos', 680);
```

La lista contiene los registros y las tuplas contienen los valores de cada
fila.

Sin embargo, existe una diferencia importante:

- En Teradata, el `INSERT` almacena filas en una tabla persistente.
- En Spark, `createDataFrame()` crea una representación distribuida dentro de
  la aplicación.
- El DataFrame no queda almacenado permanentemente por el simple hecho de
  crearlo.
- Para persistirlo sería necesario escribirlo posteriormente en un formato o
  sistema de almacenamiento.

---

### Paralelo con Informatica PowerCenter

En PowerCenter puede imaginarse como una fuente generada internamente dentro
del flujo, similar a tener registros de prueba definidos para alimentar un
mapping.

La comparación conceptual sería:

| PowerCenter | PySpark |
|---|---|
| Registro que entra al mapping | Tupla de Python |
| Conjunto de registros | Lista de tuplas |
| Puertos de la fuente | Columnas del DataFrame |
| Source Definition | Schema |
| Flujo de filas en ejecución | DataFrame |

No son equivalentes técnicamente, pero el paralelo ayuda a entender que los
datos deben tener una estructura consistente antes de ser procesados.

---

### Importancia del orden de los datos

Cuando se utiliza una lista de tuplas, Spark relaciona los valores por posición.

Por ejemplo:

```python
columns = ["customer_id", "customer_name", "credit_score"]
```

significa que Spark interpretará cada tupla así:

```text
posición 1 → customer_id
posición 2 → customer_name
posición 3 → credit_score
```

Si accidentalmente se crea esta fila:

```python
("Ana", 1, 720)
```

Spark asociará `"Ana"` con `customer_id` y `1` con `customer_name`, porque no
comprende el significado de negocio de cada valor: solamente respeta el orden
indicado.

Esta situación se parece a un `INSERT` de SQL sin nombres de columnas:

```sql
INSERT INTO customers
VALUES ('Ana', 1, 720);
```

El motor relacionará los valores con las columnas según su posición.

---

### Primera conclusión

`spark.createDataFrame()` es útil para:

- crear ejemplos pequeños;
- realizar pruebas;
- construir datos de referencia;
- transformar estructuras de Python en DataFrames;
- aprender cómo se relacionan filas, columnas y esquemas.

Para cargar grandes volúmenes productivos normalmente se utilizarán lectores
como:

```python
spark.read.csv()
spark.read.parquet()
spark.read.json()
```

Esos mecanismos se estudiarán posteriormente en este mismo notebook y con
mayor profundidad en el notebook de lectura y escritura de datos.

In [7]:
data = [
    (1, "Jhon", "Taborda", 44),
    (2, "Iris", "Perilla", 42),
    (3, "Laura", "Taborda", 14),
    (4, "Victoria", "Taborda", 13)
]

columns = ["id", "nombre", "apellido", "edad"]

df_manual = spark.createDataFrame(data, columns)

In [8]:
df_manual.show(truncate=False)
df_manual.printSchema()

+---+--------+--------+----+
|id |nombre  |apellido|edad|
+---+--------+--------+----+
|1  |Jhon    |Taborda |44  |
|2  |Iris    |Perilla |42  |
|3  |Laura   |Taborda |14  |
|4  |Victoria|Taborda |13  |
+---+--------+--------+----+

root
 |-- id: long (nullable = true)
 |-- nombre: string (nullable = true)
 |-- apellido: string (nullable = true)
 |-- edad: long (nullable = true)



In [6]:
df_manual.show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 25|
|  2|    Bob| 30|
|  3|Charlie| 35|
+---+-------+---+



## 3. Definición explícita del schema

### ¿Qué es el schema?

El `schema` describe la estructura de un DataFrame.

Define principalmente:

- el nombre de cada columna;
- el tipo de dato;
- si la columna acepta valores nulos;
- el orden de las columnas.

Cuando creamos un DataFrame usando solamente una lista de nombres:

```python
columns = ["id", "nombre", "apellido", "edad"]

df_manual = spark.createDataFrame(data, columns)
```

Spark debe analizar los valores y deducir sus tipos.

Por eso obtuvo un esquema similar a:

```text
root
 |-- id: long (nullable = true)
 |-- nombre: string (nullable = true)
 |-- apellido: string (nullable = true)
 |-- edad: long (nullable = true)
```

Spark convirtió los enteros de Python en `long` porque no le indicamos
explícitamente qué tipo queríamos utilizar.

---

### ¿Qué es `StructType`?

`StructType` representa el esquema completo de una fila.

Contiene una colección ordenada de objetos `StructField`.

Podemos imaginarlo así:

```text
StructType
├── StructField de id
├── StructField de nombre
├── StructField de apellido
└── StructField de edad
```

En términos de SQL, `StructType` se parece conceptualmente a la definición
completa de columnas de una tabla.

Por ejemplo:

```sql
CREATE TABLE personas (
    id INTEGER,
    nombre VARCHAR(100),
    apellido VARCHAR(100),
    edad INTEGER
);
```

En PySpark, esa estructura puede expresarse mediante:

```python
StructType([
    StructField("id", IntegerType(), False),
    StructField("nombre", StringType(), False),
    StructField("apellido", StringType(), False),
    StructField("edad", IntegerType(), True)
])
```

---

### ¿Qué es `StructField`?

Cada `StructField` representa una columna dentro del esquema.

Su sintaxis principal es:

```python
StructField(nombre, tipo_de_dato, nullable)
```

Sus componentes son:

- `nombre`: nombre de la columna;
- `tipo_de_dato`: tipo de dato de Spark;
- `nullable`: indica si se permiten valores nulos.

Ejemplo:

```python
StructField("edad", IntegerType(), True)
```

Significa:

- la columna se llama `edad`;
- su tipo es entero;
- puede contener valores nulos.

---

### Tipos usados en este ejemplo

```python
IntegerType()
StringType()
```

`IntegerType()` representa un número entero de 32 bits.

`StringType()` representa una cadena de caracteres.

Es importante observar que:

```python
IntegerType()
```

no es lo mismo que el tipo nativo de Python:

```python
int
```

El primero pertenece al sistema de tipos de Spark.

---

### Significado de `nullable`

Este campo:

```python
StructField("id", IntegerType(), False)
```

indica que la columna `id` no debería aceptar valores nulos.

Mientras que:

```python
StructField("edad", IntegerType(), True)
```

indica que `edad` sí puede aceptar un valor nulo.

Paralelo con SQL:

```sql
id INTEGER NOT NULL,
edad INTEGER NULL
```

Paralelo con PowerCenter:

- `nullable=False` se parece a un puerto obligatorio.
- `nullable=True` se parece a un puerto que puede recibir `NULL`.

Sin embargo, `nullable=False` no reemplaza todas las validaciones de calidad.
Debemos continuar validando los datos antes de escribirlos en una tabla
productiva.

---

### Ventajas de un schema explícito

Definir el esquema manualmente permite:

- controlar los tipos de datos;
- evitar inferencias incorrectas;
- detectar datos incompatibles;
- documentar la estructura esperada;
- mantener estabilidad entre ejecuciones;
- evitar que los tipos dependan del contenido de una muestra.

En procesos productivos es frecuente preferir un esquema explícito cuando la
estructura de la fuente es conocida.

Esto se parece a PowerCenter cuando se importa o define de manera controlada
la metadata de una Source Definition.

También se parece a Teradata, donde los tipos se establecen explícitamente en
el `CREATE TABLE` y no se deducen a partir de los datos insertados.

## Repaso rápido

### Lo que ya sé

- `SparkSession` permite trabajar con DataFrames.
- `spark.createDataFrame()` crea un DataFrame desde datos de Python.
- Cada tupla representa una fila.
- La lista de columnas asigna nombres según la posición.
- `show()` presenta registros.
- `printSchema()` muestra nombres, tipos y nulabilidad.

### Código esencial

```python
data = [
    (1, "Jhon", "Taborda", 44),
    (2, "Iris", "Perilla", 42)
]

columns = ["id", "nombre", "apellido", "edad"]

df_manual = spark.createDataFrame(data, columns)
df_manual.show()
df_manual.printSchema()

Ahora haz un ejercicio corto de refuerzo: crea un DataFrame llamado df_productos con tres productos y estas columnas:

producto_id, producto_nombre, precio

Luego ejecuta:

df_productos.show()
df_productos.printSchema()

In [9]:
data_productos = [
    (1,"Cartulina bristol 1/2 pliego",500),
    (2,"lapiz mirado N2",1000),
    (3,"Cuaderno de 100 hojas cosido gamma 1/2",3500)
]

columnas_productos = ["id","nombre","precio"]

df_productos = spark.createDataFrame(data_productos, columnas_productos)

In [10]:
df_productos.show()

+---+--------------------+------+
| id|              nombre|precio|
+---+--------------------+------+
|  1|Cartulina bristol...|   500|
|  2|     lapiz mirado N2|  1000|
|  3|Cuaderno de 100 h...|  3500|
+---+--------------------+------+



In [11]:
df_productos.printSchema()

root
 |-- id: long (nullable = true)
 |-- nombre: string (nullable = true)
 |-- precio: long (nullable = true)



### Creación de un DataFrame desde una lista de diccionarios

Otra forma de representar registros en Python es mediante diccionarios.

Ejemplo:

```python
data_clientes = [
    {
        "cliente_id": 1,
        "nombre": "Ana",
        "ciudad": "Bogotá"
    },
    {
        "cliente_id": 2,
        "nombre": "Carlos",
        "ciudad": "Medellín"
    }
]
```

En este caso:

- cada diccionario representa una fila;
- cada clave representa el nombre de una columna;
- cada valor representa el dato correspondiente a esa columna.

A diferencia de una tupla, el diccionario relaciona cada valor con una clave
explícita.

Con tuplas:

```python
(1, "Ana", "Bogotá")
```

el significado depende de la posición.

Con diccionarios:

```python
{
    "cliente_id": 1,
    "nombre": "Ana",
    "ciudad": "Bogotá"
}
```

el significado queda asociado directamente al nombre de cada campo.

#### Paralelo con SQL y Teradata

Una tupla se parece a un `INSERT` sin indicar columnas:

```sql
INSERT INTO clientes
VALUES (1, 'Ana', 'Bogotá');
```

Un diccionario se parece conceptualmente a indicar explícitamente qué valor
corresponde a cada columna:

```sql
INSERT INTO clientes (
    cliente_id,
    nombre,
    ciudad
)
VALUES (
    1,
    'Ana',
    'Bogotá'
);
```

El diccionario puede resultar más legible, aunque ocupa más código que una
tupla.

Spark puede inferir los nombres y tipos de las columnas a partir de las claves
y valores de los diccionarios.

### Ejercicio 2 — Crear un DataFrame de empleados desde diccionarios

Crear una lista llamada `data_empleados` que contenga cuatro diccionarios.

Cada diccionario debe representar un empleado con los siguientes campos:

- `empleado_id`
- `nombre`
- `cargo`
- `salario`

Después:

1. Crear un DataFrame llamado `df_empleados` utilizando
   `spark.createDataFrame()`.
2. Mostrar todos los registros sin truncar los textos.
3. Mostrar el esquema con `printSchema()`.
4. Identificar qué tipos de datos infirió Spark.
5. Explicar en una celda Markdown cuál es la diferencia entre representar una
   fila mediante una tupla y representarla mediante un diccionario.

In [13]:
data_empleados = [
     {
        "empleado_id": 1,
        "nombre": "Jhon Taborda",
        "cargo": "Ingeniero de datos",
        "salario": 5000000,
    },
    {
        "empleado_id": 2,
        "nombre": "Iris Perilla",
        "cargo": "Administradora",
        "salario": 4500000,
    },
    {
        "empleado_id": 3,
        "nombre": "Laura Taborda",
        "cargo": "Estudiante",
        "salario": 0,
    },
    {
        "empleado_id": 4,
        "nombre": "Victoria Taborda",
        "cargo": "Estudiante",
        "salario": 0,
    }
]

df_empleados = spark.createDataFrame(data_empleados)
df_empleados.show(truncate=False)
df_empleados.printSchema()

+------------------+-----------+----------------+-------+
|cargo             |empleado_id|nombre          |salario|
+------------------+-----------+----------------+-------+
|Ingeniero de datos|1          |Jhon Taborda    |5000000|
|Administradora    |2          |Iris Perilla    |4500000|
|Estudiante        |3          |Laura Taborda   |0      |
|Estudiante        |4          |Victoria Taborda|0      |
+------------------+-----------+----------------+-------+

root
 |-- cargo: string (nullable = true)
 |-- empleado_id: long (nullable = true)
 |-- nombre: string (nullable = true)
 |-- salario: long (nullable = true)



#### Resultado observado

- Cada diccionario se convirtió en una fila del DataFrame.
- Las claves de los diccionarios se utilizaron como nombres de columnas.
- No fue necesario crear una lista separada de encabezados.
- Spark infirió automáticamente los tipos de datos.
- Los campos numéricos fueron inferidos como `long`.
- Los campos de texto fueron inferidos como `string`.

#### Diferencia entre tuplas y diccionarios

Con una lista de tuplas, los valores se relacionan con las columnas según su
posición.

Con una lista de diccionarios, cada valor queda relacionado directamente con
el nombre de su clave, lo que hace el registro más explícito y legible.

### Schema explícito con `StructType` y `StructField`

Todos los DataFrames tienen un `schema`, que define su estructura:

- nombres de las columnas;
- tipos de datos;
- orden de las columnas;
- posibilidad de contener valores nulos.

Hasta ahora permitimos que Spark analizara los datos e infiriera sus tipos.

Por ejemplo:

```python
df_empleados = spark.createDataFrame(data_empleados)
```

Spark revisa los valores de los diccionarios para determinar si las columnas
son de tipo texto, entero u otro tipo compatible.

También podemos definir el esquema explícitamente utilizando:

- `StructType`: representa el esquema completo.
- `StructField`: representa una columna dentro del esquema.
- tipos como `IntegerType`, `LongType` y `StringType`.

La estructura general es:

```python
schema = StructType([
    StructField("columna_1", TipoDeDato(), nullable),
    StructField("columna_2", TipoDeDato(), nullable)
])
```

El tercer argumento de `StructField` indica si la columna permite valores
nulos:

```python
StructField("empleado_id", IntegerType(), False)
```

significa:

- la columna se llama `empleado_id`;
- almacena números enteros;
- no admite valores nulos según el esquema.

Mientras que:

```python
StructField("salario", LongType(), True)
```

indica que la columna `salario` puede aceptar valores nulos.

#### Paralelo con Teradata

Un schema explícito se parece a definir una tabla mediante `CREATE TABLE`:

```sql
CREATE TABLE empleados (
    empleado_id INTEGER NOT NULL,
    nombre VARCHAR(100) NOT NULL,
    cargo VARCHAR(100),
    salario BIGINT
);
```

En ambos casos se controla el nombre y el tipo esperado de cada columna.

Sin embargo, definir un schema en Spark no crea por sí mismo una tabla física
ni define cómo se distribuyen las filas.

#### Paralelo con Informatica PowerCenter

El schema explícito se parece a una `Source Definition` definida con metadata
controlada:

- `StructField` equivale conceptualmente a un puerto.
- El tipo de Spark equivale al tipo del puerto.
- `nullable` indica si el campo puede contener `NULL`.
- `StructType` reúne toda la definición de la fuente.

#### ¿Por qué usar un schema explícito?

Un schema explícito permite:

- controlar los tipos de datos;
- evitar inferencias inesperadas;
- documentar la estructura esperada;
- detectar datos incompatibles;
- mantener el mismo contrato entre distintas ejecuciones.

En procesos productivos, cuando la estructura de la fuente es conocida, definir
el schema explícitamente suele ser más seguro que depender de la inferencia.

In [14]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    LongType,
    StringType,
)

In [15]:
schema_empleados = StructType([
    StructField("empleado_id", IntegerType(), False),
    StructField("nombre", StringType(), False),
    StructField("cargo", StringType(), True),
    StructField("salario", LongType(), True),
])

In [16]:
df_empleados_schema = spark.createDataFrame(
    data=data_empleados,
    schema=schema_empleados,
)

df_empleados_schema.show(truncate=False)
df_empleados_schema.printSchema()

+-----------+----------------+------------------+-------+
|empleado_id|nombre          |cargo             |salario|
+-----------+----------------+------------------+-------+
|1          |Jhon Taborda    |Ingeniero de datos|5000000|
|2          |Iris Perilla    |Administradora    |4500000|
|3          |Laura Taborda   |Estudiante        |0      |
|4          |Victoria Taborda|Estudiante        |0      |
+-----------+----------------+------------------+-------+

root
 |-- empleado_id: integer (nullable = false)
 |-- nombre: string (nullable = false)
 |-- cargo: string (nullable = true)
 |-- salario: long (nullable = true)



### Ejercicio 3 — Crear un DataFrame de cuentas con schema explícito

Crear una lista de tuplas llamada `data_cuentas` con cuatro registros.

Cada cuenta debe contener:

- `cuenta_id`
- `cliente_id`
- `tipo_cuenta`
- `saldo`

Definir un schema explícito llamado `schema_cuentas` con las siguientes reglas:

- `cuenta_id`: `IntegerType`, no permite nulos.
- `cliente_id`: `IntegerType`, no permite nulos.
- `tipo_cuenta`: `StringType`, no permite nulos.
- `saldo`: `LongType`, permite nulos.

Después:

1. Crear el DataFrame `df_cuentas`.
2. Mostrar los registros sin truncar.
3. Mostrar el schema.
4. Verificar que Spark respetó los tipos y la nulabilidad definidos.

In [18]:
data_cuentas = [
    (1,2,"Cuenta de ahorros",500000),
    (2,1,"Cuenta corriente",1000000),
    (3,2,"Cuenta de inversión",2000000),
    (4,3,"Cuenta de ahorro",1500000)
]

columnas_cuentas = ["cuenta_id","cliente_id","tipo_cuenta","saldo"]

schema_cuentas = StructType([
    StructField("cuenta_id", IntegerType(), False),
    StructField("cliente_id", IntegerType(), False),
    StructField("tipo_cuenta", StringType(), False),
    StructField("saldo", LongType(), True),
])

df_cuentas = spark.createDataFrame(data_cuentas, schema=schema_cuentas)
df_cuentas.show(truncate=False)
df_cuentas.printSchema()


+---------+----------+-------------------+-------+
|cuenta_id|cliente_id|tipo_cuenta        |saldo  |
+---------+----------+-------------------+-------+
|1        |2         |Cuenta de ahorros  |500000 |
|2        |1         |Cuenta corriente   |1000000|
|3        |2         |Cuenta de inversión|2000000|
|4        |3         |Cuenta de ahorro   |1500000|
+---------+----------+-------------------+-------+

root
 |-- cuenta_id: integer (nullable = false)
 |-- cliente_id: integer (nullable = false)
 |-- tipo_cuenta: string (nullable = false)
 |-- saldo: long (nullable = true)



### Ejercicio 4 — Crear un DataFrame de transacciones con múltiples tipos de datos

Crear un DataFrame llamado `df_transacciones` utilizando una lista de tuplas y
un schema explícito.

Cada transacción debe contener los siguientes campos:

- `transaccion_id`
- `cliente_id`
- `tipo_transaccion`
- `monto`
- `fecha_transaccion`
- `fecha_hora_proceso`
- `aprobada`

Definir el schema con las siguientes reglas:

- `transaccion_id`: `LongType`, no permite nulos.
- `cliente_id`: `IntegerType`, no permite nulos.
- `tipo_transaccion`: `StringType`, no permite nulos.
- `monto`: `DecimalType(18, 2)`, no permite nulos.
- `fecha_transaccion`: `DateType`, no permite nulos.
- `fecha_hora_proceso`: `TimestampType`, no permite nulos.
- `aprobada`: `BooleanType`, no permite nulos.

Crear cuatro registros con distintos tipos de transacción, por ejemplo:

- compra;
- retiro;
- transferencia;
- pago.

Después:

1. Crear el DataFrame con `spark.createDataFrame()`.
2. Mostrar todos los registros sin truncar.
3. Mostrar el schema.
4. Confirmar que el monto quedó como `decimal(18,2)`.
5. Confirmar que la fecha quedó como `date`.
6. Confirmar que la fecha y hora quedaron como `timestamp`.
7. Confirmar que el campo `aprobada` quedó como `boolean`.

In [27]:
from datetime import date, datetime
from decimal import Decimal

from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    IntegerType,
    StringType,
    DecimalType,
    DateType,
    TimestampType,
    BooleanType,
)

data_transacciones = [
    (1, 1, "Compra", Decimal("100000.25"),date(2026, 7, 25),datetime(2026, 7, 25, 10, 30, 0), True),
    (2, 2, "Venta", Decimal("200000.50"),date(2026, 7, 26),datetime(2026, 7, 26, 11, 45, 0), False),
    (
    3,
    3,
    "Transferencia",
    Decimal("350000.75"),
    date(2026, 7, 27),
    datetime(2026, 7, 27, 9, 15, 30),
    True,
),
(
    4,
    4,
    "Pago",
    Decimal("87500.00"),
    date(2026, 7, 28),
    datetime(2026, 7, 28, 14, 20, 10),
    True,
),
    ]

schema_transacciones = StructType([
    StructField("transaccion_id", LongType(), False),
    StructField("cuenta_id", IntegerType(), False),
    StructField("tipo_transaccion", StringType(), False),
    StructField("monto", DecimalType(18, 2), False),
    StructField("fecha_transaccion", DateType(), False),
    StructField("hora_transaccion", TimestampType(), False),
    StructField("estado", BooleanType(), False)
])

df_transacciones = spark.createDataFrame(data_transacciones, schema=schema_transacciones)

In [28]:
df_transacciones.show(truncate=False)
df_transacciones.printSchema()

+--------------+---------+----------------+---------+-----------------+-------------------+------+
|transaccion_id|cuenta_id|tipo_transaccion|monto    |fecha_transaccion|hora_transaccion   |estado|
+--------------+---------+----------------+---------+-----------------+-------------------+------+
|1             |1        |Compra          |100000.25|2026-07-25       |2026-07-25 10:30:00|true  |
|2             |2        |Venta           |200000.50|2026-07-26       |2026-07-26 11:45:00|false |
|3             |3        |Transferencia   |350000.75|2026-07-27       |2026-07-27 09:15:30|true  |
|4             |4        |Pago            |87500.00 |2026-07-28       |2026-07-28 14:20:10|true  |
+--------------+---------+----------------+---------+-----------------+-------------------+------+

root
 |-- transaccion_id: long (nullable = false)
 |-- cuenta_id: integer (nullable = false)
 |-- tipo_transaccion: string (nullable = false)
 |-- monto: decimal(18,2) (nullable = false)
 |-- fecha_transacci

#### Resultado observado

- `StructType` definió el esquema completo del DataFrame.
- Cada `StructField` definió una columna.
- `DecimalType(18,2)` permitió almacenar montos con dos decimales.
- `DateType` almacenó únicamente la fecha.
- `TimestampType` almacenó fecha y hora.
- `BooleanType` almacenó valores `true` y `false`.
- Spark respetó el orden, los tipos y la nulabilidad definidos en el schema.